# 02 · Parte 2 — Modelo geométrico / estructural

**Responsable:** _(nombre)_

**Objetivo:** predecir TVT usando sólo la **geometría** (XYZ, azimut, posición en el campo) y la relación `TVT + Z` observada antes del PS y en pozos vecinos. Es la predicción "estructural": captura el dip regional y da una trayectoria base que la parte 3 refinará con el gamma ray.

Referencia: notebook `00_exploracion.ipynb`, secciones 5, 6 y 7 (`TVT + Z` cambia casi linealmente con MD y su signo depende del azimut; vecinos muy cercanos).

Convenciones comunes a las tres partes:

- Rutas relativas a la raíz del repo: `../data/raw/`, `../data/processed/`, `../models/`, `../submissions/`.
- Todo lo que una parte produce para otra se guarda en `../data/processed/<parte>/` en formato parquet, con las columnas documentadas en la sección *Salidas* de cada notebook.
- Identificador de pozo: el prefijo de 8 caracteres del archivo (`000d7d20`). Índice de punto: la fila del CSV horizontal (el mismo que usa `sample_submission`).
- Métrica oficial: RMSE sobre los puntos con índice ≥ PS (Prediction Start), en pies.

## Alcance

1. **Extrapolación propia del pozo**: estimar la tendencia de `TVT + Z` (o de TVT vs distancia horizontal) a partir del tramo conocido (< PS) y extrapolarla. Evaluar lineal, cuadrática y ventanas distintas.
2. **Dip regional a partir de vecinos**: estimar el buzamiento (gradiente de `TVT + Z` en el plano XY) con los pozos vecinos de entrenamiento, por ejemplo por regresión local ponderada por distancia o por interpolación de superficies de topes. Debe respetar los folds (sólo vecinos del train del fold).
3. **Combinación**: trayectoria base = conocido hasta el PS + dip regional aplicado a la geometría del pozo, con continuidad en el PS. Comparar contra la extrapolación propia.
4. **Modelo de residuo (opcional)**: un modelo tabular (LightGBM/CatBoost) que prediga el error de la trayectoria base a partir de features geométricas (distancia al PS, azimut, curvatura, posición XY, formación dominante…).
5. **Incertidumbre**: estimar una banda por punto (por ejemplo, desviación de los vecinos) que la parte 3 pueda usar como prior.

## Entradas

- `../data/processed/p1/wells.parquet`, `folds.parquet`, `horizontal/<well>.parquet` (de la parte 1).
- Funciones `rmse_by_well` y `validate_prediction` de la parte 1.

## Salidas (contrato con la parte 3)

| Archivo | Contenido |
|---|---|
| `../data/processed/p2/pred_geom.parquet` | `well`, `idx`, `tvt_pred`, `tvt_sigma` (predicción out-of-fold para los 773 pozos) |
| `../data/processed/p2/dip_field.parquet` | `well`, `dip_x`, `dip_y` (gradiente regional estimado para cada pozo) |
| `../models/p2/` | modelos de residuo serializados, uno por fold, si aplica |

## 1. Setup y carga de salidas de la parte 1

## 2. Extrapolación de la tendencia del propio pozo

## 3. Estimación del dip regional con pozos vecinos

## 4. Trayectoria base combinada

## 5. Modelo de residuo geométrico (opcional)

## 6. Evaluación out-of-fold y análisis de errores por azimut / zona

## 7. Exportar `pred_geom.parquet` y `dip_field.parquet`

## Checklist de entrega

- [ ] RMSE out-of-fold global y por fold reportados, comparados con la referencia constante (15.9 ft).
- [ ] La predicción es continua en el PS (sin salto respecto a `TVT_input`).
- [ ] Ningún uso de información de pozos del fold de validación al estimar el dip.
- [ ] `pred_geom.parquet` pasa `validate_prediction`.